# 02 - 数据聚合

将个体级别的科学家数据，聚合为可视化需要的国家级/学科级 JSON 文件。

## 加载数据
读取 `scientists_raw.csv`，添加派生字段。

In [1]:
import pandas as pd
import json
import os

# 切换到项目根目录
os.chdir(os.path.dirname(os.getcwd()))

df = pd.read_csv('data/scientists_raw.csv', low_memory=False)
print(f'行数: {len(df)}, 列数: {len(df.columns)}')
df.head(2)

行数: 63951, 列数: 52


,authfull,inst_name,cntry,np6019,np,firstyr,lastyr,rank (ns),nc9619 (ns),h19 (ns),...,sm-subfield-1-frac,sm-subfield-2,sm-subfield-2-frac,sm-field,sm-field-frac,rank sm-subfield-1,rank sm-subfield-1 (ns),sm-subfield-1 count,top percentile,top percentile (ns)
0,"Santamouris, Mattheos",University of New South Wales (UNSW) Australia,aus,415,415,1982,2020,4656,14845,65,...,0.447183,Energy,0.334507,Built Environment & Design,0.457746,1.0,1.0,27299.0,0.003663,0.003663
1,"Peppas, Nicholas A.",The University of Texas at Austin,usa,815,815,1973,2020,181,65035,115,...,0.326531,Polymers,0.238095,Clinical Medicine,0.344671,1.0,1.0,95625.0,0.001046,0.001046


## 派生字段

In [2]:
# 本土/海外标记（注意：cntry 列为三字母国家代码，如 'grc'/'usa'）
df['is_diaspora'] = (df['cntry'] != 'grc').astype(int)

# 学术年龄: 2020 - firstyr
df['academic_age'] = 2020 - df['firstyr']

# 百分位分组
def percentile_group(p):
    if pd.isna(p):
        return 'unknown'
    if p <= 1:
        return 'top_1'
    elif p <= 5:
        return 'top_5'
    elif p <= 10:
        return 'top_10'
    else:
        return 'other'

df['percentile_group'] = df['top percentile'].apply(percentile_group)

# 输出目录（chdir 后已在项目根目录）
OUT_DIR = 'data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

print('字段已添加: is_diaspora, academic_age, percentile_group')
print(f'本土: {(df["is_diaspora"]==0).sum()}, 海外: {(df["is_diaspora"]==1).sum()}')

字段已添加: is_diaspora, academic_age, percentile_group
本土: 35116, 海外: 28835


## 处理 cntry 缺失值

部分科学家的 `cntry`（国家）字段为 NaN。策略：

- **国家级分析**（地图、柱状图、Top 10、多维筛选）：排除这些科学家，因为不知道其所属国家就无法定位；
- **学科级分析**（树图、子学科）：保留所有科学家，因为学科字段不受国家缺失影响；
- **本土/海外对比 & 学术年龄**：也保留，因为这些分析不依赖具体国家名。

先看有多少缺失。

In [3]:
# 统计 cntry 缺失情况
nan_cntry = df['cntry'].isna().sum()
total = len(df)
print(f'总科学家: {total}')
print(f'cntry 缺失: {nan_cntry} ({nan_cntry/total*100:.2f}%)')
print(f'有国家的: {total - nan_cntry}')

# 看这些缺失国家的科学家在学科上的分布
nan_field_dist = df[df['cntry'].isna()]['sm-field'].value_counts()
print(f'\n缺失 cntry 的学科分布（前 10）:')
for field, count in nan_field_dist.head(10).items():
    print(f'  {field}: {count} 人')

# 保存缺失国家的科学家名单，供 03_exploratory_analysis.ipynb 单独分析用
df_nan_cntry = df[df['cntry'].isna()].copy()
df_nan_cntry.to_csv('data/scientists_nan_cntry.csv', index=False)
print(f'\n已保存 {len(df_nan_cntry)} 条缺失国家记录到 data/scientists_nan_cntry.csv')

总科学家: 63951
cntry 缺失: 777 (1.21%)
有国家的: 63174

缺失 cntry 的学科分布（前 10）:
  Clinical Medicine: 66 人
  Information & Communication Technologies: 23 人
  Enabling & Strategic Technologies: 11 人
  Engineering: 10 人
  Chemistry: 8 人
  Earth & Environmental Sciences: 8 人
  Biomedical Research: 7 人
  Agriculture, Fisheries & Forestry: 6 人
  Psychology & Cognitive Sciences: 5 人
  Philosophy & Theology: 3 人

已保存 777 条缺失国家记录到 data/scientists_nan_cntry.csv


## 1. 按国家聚合 → 世界地图 + Top 10 柱状图

统计每个国家的：科学家总数、海外/本土人数、中位引用、中位 h-index、Top 1% 人数、最主要学科。

> 注意：`groupby('cntry')` 默认会排除 NaN，因此 cntry 缺失的科学家不会出现在这里。这部分 ~800 人（约 1.2%）不参与国家维度分析。

In [4]:
def agg_country(g):
    top1 = g[g['percentile_group'] == 'top_1']
    # 找最主要学科（人数最多的 sm-field）
    top_field = g['sm-field'].value_counts().index[0] if not g['sm-field'].isna().all() else ''
    return pd.Series({
        'total': len(g),
        'overseas': g['is_diaspora'].sum(),
        'domestic': (1 - g['is_diaspora']).sum(),
        'overseas_pct': round(g['is_diaspora'].mean() * 100, 2),
        'median_citation': g['nc9619'].median(),
        'median_hindex': g['h19'].median(),
        'top_1_count': len(top1),
        'top_field': top_field,
        'scientist_count': len(g)
    })

by_country = df.groupby('cntry').apply(agg_country).reset_index()
by_country = by_country.rename(columns={'cntry': 'country'})
by_country = by_country.sort_values('total', ascending=False)

# 前 10 单独存一份
top10 = by_country.head(10)

by_country.to_json(f'{OUT_DIR}/by_country.json', orient='records', force_ascii=False)
top10.to_json(f'{OUT_DIR}/top10_countries.json', orient='records', force_ascii=False)

print(f'国家总数: {len(by_country)}')
print('Top 10:')
for _, r in top10[['country', 'total', 'overseas_pct']].iterrows():
    print(f'  {r["country"]}: {int(r["total"])} 人')

国家总数: 108
Top 10:
  grc: 35116 人
  usa: 9339 人
  gbr: 6165 人
  deu: 2083 人
  cyp: 1688 人
  aus: 1155 人
  fra: 1141 人
  can: 1110 人
  che: 994 人
  nld: 603 人


## 8. 仪表盘组合筛选数据

为保证前端三个筛选器（学科 / Top percentile / 本土+海外）的任意组合联动生效，
按「国家 × 学科 × 百分位 × 海外标记」四维聚合，前端根据筛选状态过滤并归总。

In [5]:
# 四维聚合：国家 × 学科 × 百分位 × 本土/海外
# 注意：cntry 缺失的科学家不参与国家级聚合（无法定位），
#       但学科级聚合会包含他们（见下方 by_field / by_subfield）
by_country_all = df.dropna(subset=['cntry']).groupby(
    ['cntry', 'sm-field', 'percentile_group', 'is_diaspora']
).agg(
    total=('authfull', 'count'),
    median_citation=('nc9619', 'median'),
    median_hindex=('h19', 'median')
).reset_index()

# 处理空值
by_country_all['sm-field'] = by_country_all['sm-field'].fillna('Unknown')
by_country_all['cntry'] = by_country_all['cntry'].str.lower()
by_country_all['is_diaspora'] = by_country_all['is_diaspora'].astype(int)
by_country_all['percentile_group'] = by_country_all['percentile_group'].astype(str)

by_country_all.to_json(f'{OUT_DIR}/by_country_all.json', orient='records', force_ascii=False)

# 同时打印缺失国家的科学家总数，方便对比
nan_cntry = df['cntry'].isna().sum()
print(f'多维聚合完成: {len(by_country_all)} 行')
print(f'  国家数: {by_country_all["cntry"].nunique()}')
print(f'  学科数: {by_country_all["sm-field"].nunique()}')
print(f'  百分位组: {by_country_all["percentile_group"].nunique()}')
print(f'  海外标记: {by_country_all["is_diaspora"].nunique()}')
print(f'  （已排除 {nan_cntry} 名 cntry 缺失的科学家）')
by_country_all.head(5)

多维聚合完成: 1408 行
  国家数: 105
  学科数: 20
  百分位组: 5
  海外标记: 2
  （已排除 777 名 cntry 缺失的科学家）


,cntry,sm-field,percentile_group,is_diaspora,total,median_citation,median_hindex
0,alb,Biomedical Research,other,1,1,347.0,12.0
1,alb,Clinical Medicine,other,1,1,84.0,4.0
2,alb,Information & Communication Technologies,top_5,1,1,1187.0,15.0
3,alb,Mathematics & Statistics,other,1,1,109.0,5.0
4,alb,Physics & Astronomy,other,1,1,111.0,7.0


## 2. 按学科聚合 → 树图

统计每个大类和子学科的人数、海外占比、Top 1% 人数、中位引用。

> 注意：学科聚合**不依赖国家字段**，因此包含所有 63,951 名科学家（含 cntry 缺失的）。

In [6]:
# 大类学科
by_field = df.groupby('sm-field').apply(
    lambda g: pd.Series({
        'scientist_count': len(g),
        'overseas_pct': round(g['is_diaspora'].mean() * 100, 2),
        'top_1_count': (g['percentile_group'] == 'top_1').sum(),
        'median_citation': g['nc9619'].median(),
        'median_hindex': g['h19'].median()
    })
).reset_index()
by_field = by_field.rename(columns={'sm-field': 'field'})
by_field = by_field.sort_values('scientist_count', ascending=False)

by_field.to_json(f'{OUT_DIR}/by_field.json', orient='records', force_ascii=False)
print(f'学科大类数: {len(by_field)}')
by_field.head(10)

学科大类数: 20


,field,scientist_count,overseas_pct,top_1_count,median_citation,median_hindex
5,Clinical Medicine,22204.0,38.05,224.0,241.0,7.0
12,Information & Communication Technologies,7007.0,45.97,120.0,114.0,5.0
10,Engineering,4476.0,48.55,164.0,153.0,6.0
15,Physics & Astronomy,4321.0,53.71,102.0,326.0,9.0
2,Biomedical Research,3968.0,55.29,44.0,362.0,8.0
9,Enabling & Strategic Technologies,3339.0,45.31,106.0,196.0,6.0
4,Chemistry,2671.0,41.22,58.0,263.0,8.0
7,Earth & Environmental Sciences,2040.0,33.73,20.0,242.5,8.0
8,Economics & Business,2037.0,52.87,19.0,143.0,6.0
0,"Agriculture, Fisheries & Forestry",1643.0,23.80,17.0,240.0,7.0


In [7]:
# 子学科（树图用）
by_subfield = df.groupby(['sm-field', 'sm-subfield-1']).apply(
    lambda g: pd.Series({
        'scientist_count': len(g),
        'overseas_pct': round(g['is_diaspora'].mean() * 100, 2),
        'top_1_count': (g['percentile_group'] == 'top_1').sum(),
        'median_citation': g['nc9619'].median(),
        'median_hindex': g['h19'].median(),
        'domestic_count': (1 - g['is_diaspora']).sum(),
        'overseas_count': g['is_diaspora'].sum()
    })
).reset_index()
by_subfield = by_subfield.rename(columns={'sm-field': 'field', 'sm-subfield-1': 'subfield'})
by_subfield = by_subfield.sort_values('scientist_count', ascending=False)

by_subfield.to_json(f'{OUT_DIR}/by_subfield.json', orient='records', force_ascii=False)
print(f'子学科数: {len(by_subfield)}')

子学科数: 846


## 3. 国家×学科矩阵 → 热力矩阵

行是国家，列是学科大类，值为科学家数量或 Top 1% 人数。

In [8]:
matrix = df.pivot_table(
    index='cntry',
    columns='sm-field',
    values='authfull',
    aggfunc='count'
).fillna(0).astype(int)

# 只保留科学家总数 >= 50 的国家
country_totals = df['cntry'].value_counts()
major_countries = country_totals[country_totals >= 50].index
matrix = matrix[matrix.index.isin(major_countries)]

result = {
    'countries': list(matrix.index),
    'fields': list(matrix.columns),
    'values': matrix.values.tolist()
}

with open(f'{OUT_DIR}/country_field_matrix.json', 'w') as f:
    json.dump(result, f, ensure_ascii=False)

print(f'矩阵维度: {len(result["countries"])} 国 × {len(result["fields"])} 学科')
print(f'国家: {result["countries"][:5]}...')

矩阵维度: 24 国 × 20 学科
国家: ['are', 'aus', 'aut', 'bel', 'bra']...


## 4. 本土/海外对比 → 散点图 + 盒须图

对比本土和海外科学家的引用、h-index、发文量分布。

In [9]:
def percentile_buckets(series, n=20):
    """计算分布的分位点，用于箱线图/山脊线图"""
    buckets = []
    for i in range(n + 1):
        buckets.append(round(series.quantile(i / n), 1))
    return buckets

def agg_diaspora(g):
    return pd.Series({
        'count': len(g),
        'median_np': g['np'].median(),
        'median_nc': g['nc9619'].median(),
        'median_h': g['h19'].median(),
        'mean_np': round(g['np'].mean(), 1),
        'mean_nc': round(g['nc9619'].mean(), 1),
        'mean_h': round(g['h19'].mean(), 1),
        'np_percentiles': percentile_buckets(g['np']),
        'nc_percentiles': percentile_buckets(g['nc9619']),
        'h_percentiles': percentile_buckets(g['h19'])
    })

diaspora_grp = df.groupby('is_diaspora').apply(agg_diaspora).reset_index()
diaspora_grp['group'] = diaspora_grp['is_diaspora'].map({0: 'domestic', 1: 'overseas'})

# 也按学科细分
diaspora_field = df.groupby(['is_diaspora', 'sm-field']).apply(
    lambda g: pd.Series({
        'count': len(g),
        'median_nc': g['nc9619'].median(),
        'median_h': g['h19'].median()
    })
).reset_index()

diaspora_grp.to_json(f'{OUT_DIR}/diaspora_comparison.json', orient='records', force_ascii=False)
diaspora_field.to_json(f'{OUT_DIR}/diaspora_by_field.json', orient='records', force_ascii=False)

print('本土 vs 海外 整体对比:')
for _, r in diaspora_grp.iterrows():
    print(f'  {r["group"]}: {int(r["count"])} 人 | 中位发文 {r["median_np"]} | 中位引用 {r["median_nc"]} | 中位h {r["median_h"]}')

本土 vs 海外 整体对比:
  domestic: 35116 人 | 中位发文 13.0 | 中位引用 169.0 | 中位h 6.0
  overseas: 28835 人 | 中位发文 13.0 | 中位引用 214.0 | 中位h 7.0


## 5. 顶尖人才迁移

Top 1% 和 Top 5% 科学家的国家分布。

In [10]:
def agg_talent(g):
    return pd.Series({
        'total': len(g),
        'overseas': g['is_diaspora'].sum(),
        'domestic': (1 - g['is_diaspora']).sum(),
        'overseas_pct': round(g['is_diaspora'].mean() * 100, 2)
    })

top_talent = df[df['percentile_group'].isin(['top_1', 'top_5'])].copy()
top_talent_by_country = top_talent.groupby('cntry').apply(agg_talent).reset_index()
top_talent_by_country = top_talent_by_country.rename(columns={'cntry': 'country'})
top_talent_by_country = top_talent_by_country.sort_values('total', ascending=False)

top_talent_by_country.to_json(f'{OUT_DIR}/top_talent_by_country.json', orient='records', force_ascii=False)
print(f'Top 1%/5% 科学家总数: {len(top_talent)}')
print(f'海外占比: {round(top_talent["is_diaspora"].mean() * 100, 1)}%')
top_talent_by_country.head(10)

Top 1%/5% 科学家总数: 4235
海外占比: 57.7%


,country,total,overseas,domestic,overseas_pct
21,grc,1790.0,0.0,1790.0,0.0
46,usa,1193.0,1193.0,0.0,100.0
20,gbr,406.0,406.0,0.0,100.0
8,can,119.0,119.0,0.0,100.0
12,cyp,113.0,113.0,0.0,100.0
3,aus,98.0,98.0,0.0,100.0
14,deu,96.0,96.0,0.0,100.0
9,che,78.0,78.0,0.0,100.0
19,fra,74.0,74.0,0.0,100.0
27,ita,30.0,30.0,0.0,100.0


## 6. 学术年龄分析 → 下一代科学家

按学术年龄分组，看不同年龄段的本土/海外分布趋势。

In [11]:
# 学术年龄分桶
df['age_group'] = pd.cut(
    df['academic_age'],
    bins=[0, 5, 10, 15, 20, 25, 30, 35, 100],
    labels=['0-5', '6-10', '11-15', '16-20', '21-25', '26-30', '31-35', '36+']
)

age_analysis = df.groupby('age_group').apply(
    lambda g: pd.Series({
        'total': len(g),
        'overseas': g['is_diaspora'].sum(),
        'domestic': (1 - g['is_diaspora']).sum(),
        'overseas_pct': round(g['is_diaspora'].mean() * 100, 2),
        'median_citation': g['nc9619'].median(),
        'median_hindex': g['h19'].median()
    })
).reset_index()

# 也按学术年龄+海外拆分的科学家数量
age_diaspora = df.groupby(['age_group', 'is_diaspora']).size().reset_index(name='count')
age_diaspora['group'] = age_diaspora['is_diaspora'].map({0: 'domestic', 1: 'overseas'})

age_analysis.to_json(f'{OUT_DIR}/academic_age.json', orient='records', force_ascii=False)
age_diaspora.to_json(f'{OUT_DIR}/age_diaspora.json', orient='records', force_ascii=False)

print('各年龄段海外占比:')
for _, r in age_analysis.iterrows():
    print(f'  {r["age_group"]}: {int(r["total"])} 人, 海外 {r["overseas_pct"]}%')

各年龄段海外占比:
  0-5: 4005 人, 海外 52.76%
  6-10: 10653 人, 海外 53.86%
  11-15: 13298 人, 海外 45.92%
  16-20: 10480 人, 海外 39.47%
  21-25: 7865 人, 海外 37.99%
  26-30: 5726 人, 海外 39.1%
  31-35: 3894 人, 海外 40.04%
  36+: 8027 人, 海外 49.25%


## 7. 全局概览统计

给 Overview Cards 用的全局指标。

In [12]:
overview = {
    'total_scientists': int(len(df)),
    'total_countries': int(df['cntry'].nunique()),
    'overseas_pct': float(round(df['is_diaspora'].mean() * 100, 2)),
    'domestic_count': int((df['is_diaspora'] == 0).sum()),
    'overseas_count': int(df['is_diaspora'].sum()),
    'top_1_count': int((df['percentile_group'] == 'top_1').sum()),
    'median_citation': float(round(df['nc9619'].median(), 1)),
    'median_hindex': float(round(df['h19'].median(), 1)),
    'total_fields': int(df['sm-field'].nunique()),
    'total_subfields': int(df['sm-subfield-1'].nunique())
}

with open(f'{OUT_DIR}/overview_stats.json', 'w') as f:
    json.dump(overview, f, ensure_ascii=False)

for k, v in overview.items():
    print(f'{k}: {v}')

total_scientists: 63951
total_countries: 108
overseas_pct: 45.09
domestic_count: 35116
overseas_count: 28835
top_1_count: 927
median_citation: 186.0
median_hindex: 6.0
total_fields: 20
total_subfields: 174


## 8. 导出散点图采样 & 复制到 web/data/

为 D3 前端生成采样数据（全量 6 万点浏览器扛不住），并将所有 JSON 复制到 web/data/。

In [13]:
import random
import shutil

# 采样 3000 个点用于散点图
sample_df = df[(df['np'] > 0) & (df['nc9619'] > 0)].sample(n=min(3000, len(df)), random_state=42)
scatter_sample = sample_df[['np', 'nc9619', 'is_diaspora']].to_dict(orient='records')
with open(f'{OUT_DIR}/scatter_sample.json', 'w') as f:
    json.dump(scatter_sample, f, ensure_ascii=False)

# 复制到 web/data/
WEB_DATA_DIR = 'web/data'
os.makedirs(WEB_DATA_DIR, exist_ok=True)
for fname in os.listdir(OUT_DIR):
    if fname.endswith('.json'):
        shutil.copy2(os.path.join(OUT_DIR, fname), os.path.join(WEB_DATA_DIR, fname))

print('已复制到 web/data/:')
for f in sorted(os.listdir(WEB_DATA_DIR)):
    size = os.path.getsize(os.path.join(WEB_DATA_DIR, f))
    print(f'  {f:35s} {size/1024:>8.1f} KB')

已复制到 web/data/:
  academic_age.json                        1.1 KB
  age_diaspora.json                        1.1 KB
  by_country.json                         20.3 KB
  by_country_all.json                    207.3 KB
  by_field.json                            2.8 KB
  by_subfield.json                       178.8 KB
  country_field_matrix.json                2.5 KB
  diaspora_by_field.json                   4.0 KB
  diaspora_comparison.json                 1.0 KB
  overview_stats.json                      0.2 KB
  scatter_sample.json                    130.4 KB
  top10_countries.json                     1.9 KB
  top_talent_by_country.json               3.8 KB
